# Does simulated disagreement predict real controversy?

A backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish) that scores something every previous run threw away — and that already broke once, which is why this notebook calibrates instead of guessing.

## Why this exists

Every earlier backtest reduced a finished simulation down to `sign(final mean opinion)` — one bit, the same bit a single LLM call produces. Scored that way, at n=200, the result was unambiguous:

| | accuracy |
|---|---|
| author-karma heuristic | 62.5% |
| **simulation** | **51.5%** |
| single LLM call | 50.0% |
| majority class | 50.0% |

p = 0.9994. The simulation is at chance, beating one raw model call by all of 1.5 points — a real but negligible contribution from all the agents and rounds put together.

Here's the problem with concluding "the engine is worthless" from that number: we evaluated a population on the one output where a population has no structural advantage. A multi-agent run also produces a distribution — 24 opinions with a spread to them — and nothing about the mean captures whether that crowd actually agreed with itself. This notebook tests that other moment: does the dispersion of simulated opinion predict whether a real HN thread turned into an argument?

## What went wrong the first time, and why this version is different

The first attempt at this (n=107) reported 53.3% accuracy and looked unremarkable, right up until two problems surfaced. First, a real bug: `CachingAdapter`, which the CLI wraps every HN run in, never got updated to delegate the new dispersion-scoring hook, so it silently fell back to scoring the mean axis all over again. That 53.3% never measured controversy at all — fixed now, with `CachingAdapter` delegating every `DomainAdapter` method and a structural test enforcing it stays that way. Second, a guessed threshold: the dispersion cutoff (`stddev >= 0.35`) was a bare guess with nothing behind it, and the real stddev range on that run turned out to be 0.085–0.286 — never once reaching 0.35. A threshold that never fires isn't a test, correctly-scored axis or not.

This notebook fixes both. It calibrates the threshold from a held-out calibration batch — the median of their simulated stddevs — and only reports accuracy on a disjoint evaluation batch, so the threshold gets chosen before, and independently of, the data it's scored against (METHODOLOGY.md rules 3 and 6 cover why that separation matters).

## Honest framing before you run it

None of this is a claim that a single call couldn't make the same prediction in principle — you can just ask a model "will this be controversial," which is exactly what the `single_llm` rung does. The distinction is that the simulation *derives* disagreement from population heterogeneity rather than asserting it directly, which is real but narrower than "structurally impossible for one call." Controversy prediction itself also isn't a novel task — it's done on Reddit and Wikipedia edit wars already — so doing it on HN isn't the interesting part here. Given the mean axis is already settled at chance, the honest expectation is that this fails too; the point of running it is closing the question of whether the standard evaluation was simply measuring the wrong output, not rescuing a result.


---
## What the data is

The source is the [Hacker News Algolia API](https://hn.algolia.com/api) — free, unauthenticated, roughly 10k requests an hour, so no key and no scraping.

The sample is settled stories at least 24 hours old, pulled class-balanced (half above the high points threshold, half below the low one). What each agent reads is strictly submission-time fields, never the outcome:

| Field | Example |
|---|---|
| title | "China is now the world's greatest oil power" |
| author + karma | `bookofjoe` (110,566) |
| url domain | `economist.com` |
| type | story / Ask HN / Show HN |
| self-text | first 500 chars, if any |

The label comes from `num_comments / points` at settlement: a ratio of 0.7 or above counts as contested (a thread arguing with itself), under 0.4 is consensus (quietly upvoted), and anything in between gets skipped as too ambiguous. Stories under 20 points are skipped outright too — zero comments on a 1-point story means nobody saw it, not that everyone quietly agreed.

That floor plus the mid-ratio gap remove most of what gets pulled — expect roughly one in three stories to end up usable, which is why `PULL_LIMIT` below is set large. Point-in-time safety is enforced in code: the seed enricher lives in a separate module from the ground-truth fetcher, and tests assert the target values never leak into the seed text or metadata.


---
## What the model is

Inference runs on [qwen2.5:7b](https://ollama.com/library/qwen2.5) (Q4_K_M, ~4.7 GB), served locally by Ollama inside the notebook — no API keys, $0. It fits comfortably in a T4's 16 GB of VRAM, which is the whole reason this runs here: the same job on a loaded 16 GB CPU box managed zero completed events in 80 minutes.

The simulation itself runs a population of 24 agents over 3–4 rounds, split into three tiers each round:

| Tier | Share | What happens |
|---|---|---|
| T1 originators | ~10% | LLM writes a structured post |
| T2 reactors | ~20% | LLM re-evaluates after reading a feed of others' posts |
| T3 drifters | rest | deterministic herding maths, no LLM call |

The agents are six HN archetypes with different resistance, recency bias, contrarian tendency and herding coefficients — heterogeneity is the whole point, since a population of identical agents would converge trivially and its spread would carry no information at all.

What actually gets scored is the standard deviation of the 24 final opinions, against a threshold calibrated by this run rather than hardcoded. And as with every backtest in this repo, the simulation has to beat every rung of the baseline ladder to count for anything:

| Rung | What it controls for |
|---|---|
| majority class | degenerate data |
| `naive` | Ask-HN / question-mark heuristic, no model |
| `single_llm` | one call asked the same controversy question |
| the simulation | — |


---
## 1. Setup

Sidebar: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama — its installer needs it to extract, Kaggle's base image doesn't ship it, and skipping this step makes the install fail quietly. It only surfaces later as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms the CachingAdapter delegation fix
# and the calibrated-threshold code are actually present in this clone.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

---
## 2. Configuration

`PULL_LIMIT` needs real headroom: the 20-point floor and the mid-ratio gap together discard roughly two of every three pulled stories, and the calibration/evaluation split then halves whatever's left again. Aim for at least ~40 usable events on each side of that split.


In [ ]:
PULL_LIMIT = 300      # stories to pull; expect ~1 in 3 to be scoreable
N_AGENTS   = 24        # population size - the spread of THIS is what's scored
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}")

### Throughput check

Roughly 26 model calls per event. Worth confirming the per-call cost before committing to the full run — double digits here means you're on CPU regardless of what the assertion above claimed.


In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event")

---
## 3. Run it


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/hn_controversy_calibrated.log


### Reading the log

The log walks through the same shape every calibrated-controversy run does: how many pulled events actually had a controversy direction (the points floor and gap zone will have discarded most of them), how the calibration/evaluation split came out (either half under ~15 isn't enough basis to trust), and the derived threshold — worth comparing against the 0.35 that failed to fire at all last time. The report block at the end is the part that matters: `beats_baselines` needs to PASS on every rung, and `p_value_vs_best` needs to be well under 0.05, for any of this to be a result rather than noise. If you see a warning about a constant predictor on the eval set, that means the two halves ended up with different stddev distributions, or n is still too small — read that as inconclusive, not as a negative finding.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache

The cache holds every simulated run's final distribution, so a later question about these same events — a different threshold, a different axis — costs nothing to re-score.

## Result interpretation

A negative here is the expected outcome and still worth recording. The mean axis is already settled at chance across n=200, and if a properly calibrated dispersion axis also turns up nothing, that closes the question of whether the standard evaluation was measuring the wrong output all along — cleanly this time, with no bug or guessed threshold muddying the answer. Either way the number belongs in [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md), alongside the rest.
